# Seeing Trees

A Python tutorial on trees, boosting and the ROC curve. It does four things:

- Grows regression trees in a simulated world where we know the truth, and shows that each yes/no question is one cut of a map.
- Chooses a tree's depth the honest way: **train fits, validation picks, test grades once.**
- Shows boosting at work: many small trees, added one at a time.
- Ranks real firms by bankruptcy risk with a classification tree, and draws the ROC curve by hand.

**Before you run anything**

1. Download `polish_bankruptcy.csv.gz` from Brightspace. It is the only file this notebook needs.
2. In Colab, click the folder icon on the left edge and drag the file into the Files pane.
3. Run the **Setup** cell and the **helpers** cell (Shift + Enter), then every cell in order.

- Colab deletes uploaded files when a session ends. Coming back later? Upload the file again, then run Setup and the helpers again.

## Setup

- Run this first, every session.
- It fetches the course's data helper, loads the libraries, checks that the data file is there, and sets large type for the figures.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

# Check now that the data file is there, so a missing upload shows up at once.
bankruptcy_file = resolve_data_path("polish_bankruptcy.csv.gz")

# Large type for the screen, and the course's colors (Okabe-Ito: safe for color-blind readers).
plt.rcParams.update({"font.size": 16, "axes.titlesize": 17, "figure.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, VERMILLION, GREEN, GRAY = "#0072b2", "#d55e00", "#009e73", "#777777"
print("Setup done. Data file found:", bankruptcy_file.name)

- Then run this cell too. It defines helpers that draw a fitted tree as a flowchart and shade a map of two characteristics by a model's forecast.
- You need not read them.

In [ ]:
SIZE_VALUE = ["size", "value"]     # the two characteristics the maps show
# A fine grid of points covering the map: size across, value up, both from -1 to +1.
SIZE_GRID, VALUE_GRID = np.meshgrid(np.linspace(-1, 1, 201), np.linspace(-1, 1, 201))
MAP_POINTS = pd.DataFrame({"size": SIZE_GRID.ravel(), "value": VALUE_GRID.ravel()})


def leaf_numbers(tree):
    """Number a fitted tree's final leaves 1, 2, 3, ... from left to right in its flowchart."""
    nodes = tree.tree_
    final_leaves = [node for node in range(nodes.node_count) if nodes.children_left[node] == -1]
    return {node: number for number, node in enumerate(final_leaves, start=1)}


def describe_question(tree, node):
    """The question a fitted tree asks at one node, in words, such as 'size ≤ 0.02?'."""
    name = tree.feature_names_in_[tree.tree_.feature[node]]
    cut = tree.tree_.threshold[node]
    return f"{name} ≤ {cut:.2f}?".replace("-", "−")


def draw_tree(tree, axis):
    """Draw a fitted tree as a flowchart: yes goes left, no goes right. Each final leaf shows its
    number and its forecast: the average return of the training rows that land in it."""
    nodes = tree.tree_
    depth = tree.get_depth()
    numbers = leaf_numbers(tree)
    font = 15 if depth <= 2 else 13
    spots = {}                                        # where each node sits: (across, up)

    def place(node, left, right, level):
        spots[node] = ((left + right) / 2, 0.92 - 0.82 * level / depth)
        if nodes.children_left[node] != -1:           # a question: place its two branches below
            middle = (left + right) / 2
            place(nodes.children_left[node], left, middle, level + 1)
            place(nodes.children_right[node], middle, right, level + 1)

    place(0, 0.0, 1.0, 0)
    for node, (across, up) in spots.items():
        if nodes.children_left[node] == -1:           # a final leaf: its number and its forecast
            forecast = 100 * nodes.value[node][0][0]
            axis.text(across, up, f"leaf {numbers[node]}\n{forecast:.2f}%".replace("-", "−"),
                      ha="center", va="center", fontsize=font,
                      bbox=dict(boxstyle="square", facecolor="#eeeeee", edgecolor="gray"))
            continue
        axis.text(across, up, describe_question(tree, node), ha="center", va="center",
                  fontsize=font, bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"))
        for child, word in [(nodes.children_left[node], "yes"), (nodes.children_right[node], "no")]:
            axis.annotate("", xy=spots[child], xytext=(across, up),
                          arrowprops=dict(arrowstyle="->", linewidth=1.5, shrinkA=12, shrinkB=22))
            middle_across = (across + spots[child][0]) / 2
            middle_up = (up + spots[child][1]) / 2
            axis.text(middle_across + (-0.015 if word == "yes" else 0.015), middle_up, word,
                      ha="right" if word == "yes" else "left", va="center",
                      fontsize=font - 1, color="dimgray")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_axis_off()


def format_map(axis):
    """Axis labels and ticks for a map of size (across) and value (up)."""
    axis.set_xticks([-1, 0, 1], ["−1\nsmall", "0", "+1\nbig"])
    axis.set_yticks([-1, 0, 1], ["−1", "0", "+1"])
    axis.set_xlabel("size")
    axis.set_ylabel("value")


def map_forecasts(model):
    """A fitted model's forecast at every point of the map, in % a month."""
    return 100 * model.predict(MAP_POINTS).reshape(SIZE_GRID.shape)


def shade_map(forecasts, axis, limit=4.5):
    """Shade the map by forecasts in % a month: blue high, red low. Returns the shading,
    which a color bar needs."""
    shading = axis.pcolormesh(SIZE_GRID, VALUE_GRID, forecasts, cmap="RdBu",
                              vmin=-limit, vmax=limit, shading="auto")
    format_map(axis)
    return shading


def label_leaves(tree, axis):
    """Write each leaf's number in the middle of its region of the map (leaves too thin to hold
    a label are left unlabeled)."""
    leaves = tree.apply(MAP_POINTS).reshape(SIZE_GRID.shape)   # the leaf each map point lands in
    font = 14 if tree.get_depth() <= 2 else 13
    for node, number in leaf_numbers(tree).items():
        inside = leaves == node
        if inside.mean() > 0.015:
            axis.text(SIZE_GRID[inside].mean(), VALUE_GRID[inside].mean(), f"leaf {number}",
                      ha="center", va="center", fontsize=font,
                      bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))

## 1. A World We Control

- A simulated stock market, so we know the true rule and can check what a tree finds. It is the world of the week 4 slides' "A World We Control".
- **One row is one stock in one month** (a stock-month).
- Six characteristics, each spread evenly between −1 and +1:
  - **size**: −1 the smallest firms (by market value), +1 the biggest;
  - **value**: +1 the cheapest stocks relative to their accounting value (high book-to-market), −1 the most expensive ("growth");
  - **momentum**: +1 the stocks that rose most over the past year;
  - **noise 1** to **noise 3**: pure noise, unrelated to returns.

In [ ]:
# A simulated world of stocks, one row per stock-month. Six characteristics, each spread
# evenly between -1 and +1: size, value, momentum and three columns of pure noise.
CHARACTERISTICS = ["size", "value", "momentum", "noise 1", "noise 2", "noise 3"]


def make_world(n_rows, generator, signal=0.06):
    """n_rows stock-months. Two rules: value pays only among small firms (size below 0), and
    momentum pays a little everywhere. Returns are simple, in decimals per month, with noise
    of 10% a month (sd)."""
    world = pd.DataFrame(generator.uniform(-1, 1, size=(n_rows, 6)), columns=CHARACTERISTICS)
    value_pays = np.where(world["size"] < 0, world["value"], 0.0)
    world["expected"] = signal * value_pays + signal / 3 * world["momentum"]
    world["return"] = world["expected"] + generator.normal(0, 0.10, size=n_rows)
    return world


generator = np.random.default_rng(2026)      # a fixed seed: the same world on every run
stocks = make_world(15000, generator)        # 15,000 stock-months
stocks.head().round(3)

**The true rule**, in the `expected` column (decimals per month: 0.03 = 3%):

- Among **small** firms (size below 0), the expected return is 6% × value: from −6% a month for the most expensive stocks to +6% for the cheapest.
- Among **big** firms, value does not matter.
- Momentum adds up to ±2% a month, for every firm. The noise columns add nothing.
- `return` is the expected return plus noise. The noise has a standard deviation of 10% a month, far larger than the rule.
- The models never see `expected`: we keep it only to check them.

**The figure.** Two maps with size across and value up.

- Left: the true rule (momentum set to 0). Blue is a high expected return, red a low one, white zero; the color bar gives the scale.
- Right: 1,500 of the stocks, each dot colored by its actual return.
- The rule is in there, but buried in noise. The right panel is all a model gets to see.

In [ ]:
# Left: the true expected return on the map (momentum at 0), in % a month.
truth_on_map = 100 * 0.06 * np.where(SIZE_GRID < 0, VALUE_GRID, 0)
# Right: the first 1,500 stock-months, each colored by its actual return, in % a month.
sample = stocks.head(1500)

figure, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
shading = axes[0].pcolormesh(SIZE_GRID, VALUE_GRID, truth_on_map, cmap="RdBu",
                             vmin=-6, vmax=6, shading="auto")
figure.colorbar(shading, ax=axes[0], label="expected return, % a month")
label_style = dict(ha="center", va="center", bbox=dict(boxstyle="round", facecolor="white"))
axes[0].text(-0.5, 0, "small firms:\nvalue pays", **label_style)
axes[0].text(0.5, 0, "big firms:\nvalue does not", **label_style)
axes[0].set_title("the true rule")
dots = axes[1].scatter(sample["size"], sample["value"], c=100 * sample["return"],
                       cmap="RdBu", vmin=-20, vmax=20, s=14)
figure.colorbar(dots, ax=axes[1], label="actual return, % a month")
axes[1].set_title("what a model sees")
for axis in axes:
    format_map(axis)
plt.show()

## 2. See One Tree

- A regression tree asks yes/no questions about the characteristics, one at a time.
- Each question cuts the map in two along one straight line. After the last question, every stock-month sits in one **leaf**.
- **The forecast for a leaf is the average return of the training rows in it.** Every stock in a leaf gets that same forecast.
- The trees here see only two characteristics, size and value, so each tree fits on one map.
- They learn from the first 6,000 stock-months only. The other 9,000 stay unseen until "Depth Is the Dial".

In [ ]:
train = stocks.iloc[:6000]        # the first 6,000 stock-months: the rows the trees learn from


def show_tree(depth):
    """Fit a tree of this depth on size and value, then draw it two ways, side by side."""
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    tree.fit(train[SIZE_VALUE], train["return"])
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.4), constrained_layout=True,
                                gridspec_kw={"width_ratios": [1.45, 1]})
    draw_tree(tree, axes[0])
    axes[0].set_title(f"depth {depth}: the questions")
    shading = shade_map(map_forecasts(tree), axes[1])
    label_leaves(tree, axes[1])
    axes[1].set_title("the same leaves on the map")
    figure.colorbar(shading, ax=axes[1], label="forecast, % a month")
    plt.show()
    return tree

**Depth 1: one question.**

- Left: the flowchart. "Yes" means the stock's number is at or below the cut.
- Right: the same tree as a map. The one question is one straight cut; each side is a leaf.
- Each leaf's color is its forecast, the number printed in the flowchart: blue high, red low.

In [ ]:
tree_1 = show_tree(depth=1)

**Depth 2: one more question on each side.**

- Four leaves. A value question cuts across the map (yes = below the cut); a size question cuts up and down (yes = left of the cut).
- Look for the rule we hid: the tree asks about value first, then about size on both sides, and cuts both near 0.
- Small firms: one red leaf and one blue leaf. Big firms: two leaves near zero.

In [ ]:
tree_2 = show_tree(depth=2)

The same depth-2 tree as text rules, one line per question:

- `export_text` calls a leaf's forecast its "value"; we rename it "forecast", so it is not confused with the value characteristic.
- Forecasts are in decimals: 0.032 is 3.2% a month.

In [ ]:
rules = export_text(tree_2, feature_names=SIZE_VALUE, decimals=3)
print(rules.replace("value: [", "forecast: ["))

**Depth 3: eight leaves.**

- Among small firms (left half), the third questions split by value again. There, the return keeps rising with value, and more leaves make a finer staircase.
- Among big firms (right half), the third questions chase noise. One cuts off a thin stripe near value 0, too thin to label on the map: its forecast rests on a handful of stock-months. Find it in the flowchart: the most extreme forecast of all eight.
- More depth means more and smaller leaves, and the averages of small leaves are noisy.

In [ ]:
tree_3 = show_tree(depth=3)

## 3. Depth Is the Dial

- A deeper tree fits its training rows better. Does it forecast new rows better?
- Depth is a tuning parameter. We choose it with rows the tree did not learn from.
- Three blocks of rows, three jobs: **train fits, validation picks, test grades once.**
  - **Train fits:** each candidate tree learns from the training block.
  - **Validation picks:** the depth with the lowest error on the validation block wins.
  - **Test grades once:** the test block scores the finished choice once, at the end. Nothing is chosen with it.
- The rows of this world are in random order, so cutting the table by position gives fair blocks. With returns through time, the later blocks must come later in time.

In [ ]:
# Three blocks of the 15,000 stock-months.
train = stocks.iloc[:6000]               # the first 6,000: the rows the trees above learned from
validation = stocks.iloc[6000:10000]     # the next 4,000
test = stocks.iloc[10000:]               # the last 5,000: set aside until the very end
print(f"train {len(train):,} rows, validation {len(validation):,} rows, "
      f"test {len(test):,} rows")

- **Error** here is the mean squared error, with returns in percent: a miss of 10% counts 100. Lower is better.
- The noise alone has a standard deviation of 10%, so even the true rule scores about 100.

In [ ]:
def squared_error(actual, forecasts):
    """Mean squared error, with returns in percent (they are stored in decimals, so x 100)."""
    misses = 100 * (actual - forecasts)
    return np.mean(misses ** 2)


depths = list(range(1, 13))
train_errors, validation_errors = [], []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0)
    tree.fit(train[SIZE_VALUE], train["return"])                                  # train fits
    train_errors.append(squared_error(train["return"], tree.predict(train[SIZE_VALUE])))
    validation_errors.append(squared_error(validation["return"],
                                           tree.predict(validation[SIZE_VALUE])))

# Validation picks: the depth with the lowest validation error.
best_depth = depths[int(np.argmin(validation_errors))]
# A benchmark: forecast every stock with the training block's average return.
average_error = squared_error(validation["return"], train["return"].mean())
print("depth with the lowest validation error:", best_depth)

**The figure.** Error against the depth of the tree.

- Blue, "train": the error on the rows the tree learned from. It falls with every level.
- Orange, "validation": the error on rows it never saw. It falls, then rises: deep trees memorize noise.
- Gray dashed: forecasting every stock with the training average. Deep trees end up worse than that.
- The dotted line marks the depth validation picks.

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(depths, train_errors, color=BLUE, marker="o", label="train: rows it learned from")
plt.plot(depths, validation_errors, color=VERMILLION, marker="o",
         label="validation: rows it never saw")
plt.axhline(average_error, color=GRAY, linestyle="--", label="the training average")
plt.axvline(best_depth, color="black", linestyle=":", linewidth=2)
plt.text(best_depth + 0.15, max(validation_errors), f"validation picks\ndepth {best_depth}",
         va="top")
plt.xticks(depths)
plt.xlabel("depth of the tree")
plt.ylabel("mean squared error\n(returns in %)")
plt.legend(loc="lower left", framealpha=1, edgecolor="none")   # a white box over the dotted line
plt.show()

**Test grades once.**

- The finished choice: the tree of the picked depth, fitted on the training block.
- The test block scores it, once.
- Refitting on train and validation together, with the depth already fixed, is also common: the review video on tuning parameters does it. Here the tree keeps its training-block fit, so each block does one job.

In [ ]:
chosen_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=0)
chosen_tree.fit(train[SIZE_VALUE], train["return"])
# Test grades once: the only time the test block is used.
test_error = squared_error(test["return"], chosen_tree.predict(test[SIZE_VALUE]))
test_average_error = squared_error(test["return"], train["return"].mean())
print(f"test error, the depth-{best_depth} tree: {test_error:.1f}")
print(f"test error, the training average: {test_average_error:.1f}")
print(f"out-of-sample R² of the tree: {100 * (1 - test_error / test_average_error):.1f}%")

- That is the honest grade: no choice was made with the test block.
- Had we picked the depth by its test score, the grade would flatter the tree, because the pick would fit the test block's noise too.

## 4. Boosting, Seen

- One tree is coarse: a few leaves, one forecast each.
- **Boosting** adds many small trees, one at a time:
  1. Start: every forecast is the training average.
  2. Fit a small tree to the **leftovers**: what the forecast still gets wrong.
  3. Add a small fraction of that tree's forecast (the **learning rate**, 0.1 here). Repeat from step 2.
- scikit-learn's `GradientBoostingRegressor` runs this loop. Its trees here have depth 2, so each can ask about size and then about value.
- Again on size and value only, and fitted on the training block only.

In [ ]:
# 500 small trees of depth 2; each adds a tenth of its forecast (learning rate 0.1).
boosted = GradientBoostingRegressor(max_depth=2, learning_rate=0.1, n_estimators=500,
                                    random_state=0)
boosted.fit(train[SIZE_VALUE], train["return"])                                   # train fits

# staged_predict hands out the forecasts after 1 tree, 2 trees, ..., 500 trees: one fit.
snapshots = {}
for count, forecasts in enumerate(boosted.staged_predict(MAP_POINTS), start=1):
    if count in (1, 10, 100):
        snapshots[count] = 100 * forecasts.reshape(SIZE_GRID.shape)    # in % a month

**The figure.** The boosted forecast on the map after 1, 10 and 100 trees.

- Colors as before: blue high, red low, on the scale of the color bar. Each title gives that panel's lowest and highest forecast.
- After 1 tree the map is nearly white: one tree adds only a tenth of a step.
- After 10 trees the hidden rule shows: on the left, the forecast rises with value, in many small steps; on the right, it stays near zero.
- After 100 trees the rule is sharper, but thin streaks appear, mostly at the edges: later trees start chasing single noisy stock-months. The extreme forecasts in that panel's title come from those streaks.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4.9), constrained_layout=True)
for axis, count in zip(axes, [1, 10, 100]):
    shading = shade_map(snapshots[count], axis)
    word = "tree" if count == 1 else "trees"
    lowest, highest = snapshots[count].min(), snapshots[count].max()
    axis.set_title(f"after {count} {word}\nforecasts {lowest:.1f}% to {highest:.1f}%"
                   .replace("-", "−"))
figure.colorbar(shading, ax=axes, label="forecast, % a month", shrink=0.9)
plt.show()

**How many trees?** The number of trees is a dial too, so validation picks it.

- `staged_predict` again: the error after each number of trees, on the training block and on the validation block.

In [ ]:
counts = list(range(1, 501))
train_errors_by_count, validation_errors_by_count = [], []
for forecasts in boosted.staged_predict(train[SIZE_VALUE]):
    train_errors_by_count.append(squared_error(train["return"], forecasts))
for forecasts in boosted.staged_predict(validation[SIZE_VALUE]):
    validation_errors_by_count.append(squared_error(validation["return"], forecasts))

# Validation picks: the number of trees with the lowest validation error.
best_count = counts[int(np.argmin(validation_errors_by_count))]
print("number of trees with the lowest validation error:", best_count)

**The figure.** Error against the number of trees, read as the depth figure.

- Blue, "train": falls the whole way, as later trees fit the noise in the training rows.
- Orange, "validation": falls fast, flattens, then creeps up.
- The dotted line marks the number of trees validation picks.

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(counts, train_errors_by_count, color=BLUE, label="train: rows it learned from")
plt.plot(counts, validation_errors_by_count, color=VERMILLION,
         label="validation: rows it never saw")
plt.axhline(average_error, color=GRAY, linestyle="--", label="the training average")
plt.axvline(best_count, color="black", linestyle=":", linewidth=2)
plt.text(best_count + 8, max(train_errors_by_count), f"validation picks\n{best_count} trees",
         va="top")
plt.ylim(min(train_errors_by_count) - 1, average_error + 1.5)
plt.xlabel("number of trees")
plt.ylabel("mean squared error\n(returns in %)")
plt.legend(loc="center right", framealpha=1, edgecolor="none")
plt.show()

print(f"validation error, boosting with {best_count} trees: "
      f"{validation_errors_by_count[best_count - 1]:.2f}")
print(f"validation error, the best single tree (depth {best_depth}): "
      f"{min(validation_errors):.2f}")

- Validation picks fewer than 100 trees: the streaks in the 100-tree map were already noise.
- On this simple rule, boosting and the best single tree are about equal on validation: both come close to what size and value can tell us.
- The test block stays closed. It graded one finished choice, and nothing was chosen with it.

## 5. Classification and the ROC Curve

- Now the outcome is yes or no: did a firm go bankrupt? A **classification tree** asks the same kind of yes/no questions.
- Its forecast for a leaf is **the share of the leaf's training firms that failed**. That is each firm's **risk score**.
- The data: Polish manufacturing firms (Zieba, Tomczak and Tomczak, 2016).
  - **One row is one firm**: 64 financial ratios from its accounts (`attr1` to `attr64`: profit, debt, liquidity, turnover), and `bankrupt`, 1 if the firm went bankrupt within `horizon` years.
  - `horizon` runs from 1 to 5 years. Each horizon is a different set of firms.
- **We use horizon 3: did the firm go bankrupt within three years?**

In [ ]:
# Polish manufacturing firms (Zieba, Tomczak and Tomczak, 2016; UCI Machine Learning
# Repository, CC BY 4.0). One row is one firm: 64 financial ratios from its accounts, and
# bankrupt = 1 if it went bankrupt within `horizon` years. Each horizon is a different set
# of firms.
firms = pd.read_csv(bankruptcy_file)
ratio_columns = [f"attr{number}" for number in range(1, 65)]
three_years = firms[firms["horizon"] == 3]
print(f"{len(three_years):,} firms; {three_years['bankrupt'].sum()} went bankrupt within "
      f"three years ({100 * three_years['bankrupt'].mean():.1f}%)")

- Hold out 30% of the firms. The tree never sees them; they grade its ranking.
- The depth is fixed at 4 here, so there is nothing to pick and no validation block.

In [ ]:
# stratify keeps the same share of failing firms in both groups; random_state fixes the draw.
train_firms, test_firms = train_test_split(three_years, test_size=0.3,
                                           stratify=three_years["bankrupt"], random_state=0)

# A few ratios are blank (a zero or missing denominator). Fill each blank with that ratio's
# median among the TRAINING firms, so the held-out firms play no part in fitting.
training_medians = train_firms[ratio_columns].median()
ratios_train = train_firms[ratio_columns].fillna(training_medians)
ratios_test = test_firms[ratio_columns].fillna(training_medians)
failed_train = train_firms["bankrupt"].to_numpy()      # 1 = went bankrupt, 0 = healthy
failed_test = test_firms["bankrupt"].to_numpy()
print(f"training firms {len(train_firms):,}, held-out firms {len(test_firms):,} "
      f"({failed_test.sum()} of them failed)")

In [ ]:
# A classification tree of depth 4; every leaf keeps at least 30 training firms.
risk_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=0)
risk_tree.fit(ratios_train, failed_train)

# predict_proba gives two columns: the chance of 0 (healthy) and of 1 (failed). Keep the second.
risk_scores = risk_tree.predict_proba(ratios_test)[:, 1]
print("risk scores of the first five held-out firms:", np.round(risk_scores[:5], 3))
print("number of leaves:", risk_tree.get_n_leaves(),
      "| number of different risk scores:", len(np.unique(risk_scores)))

- Every firm in a leaf gets the same risk score, so the tree has only as many different scores as leaves.

### Accuracy Misleads

- Few firms fail. A "model" that says "nobody fails" is right for almost every firm, and catches no failing firm at all.
- Compare it with the tree, which flags a firm when its risk score is above 50%.

In [ ]:
share_failed = failed_test.mean()
print(f"'nobody fails': right for {100 * (1 - share_failed):.1f}% of firms, "
      f"catches 0 of {failed_test.sum()} failing firms")

flagged = risk_scores > 0.5                       # the tree's yes/no call at a 50% cutoff
accuracy = np.mean(flagged == (failed_test == 1))
caught = np.sum(flagged & (failed_test == 1))
print(f"the tree at 50%: right for {100 * accuracy:.1f}% of firms, "
      f"catches {caught} of {failed_test.sum()} failing firms")

- Accuracy hardly tells the two apart, though only the tree finds failing firms.
- So we **rank** firms from riskiest to safest and ask: how well does the ranking put failing firms first?

### The ROC Curve, by Hand

- Sort the firms from the highest risk score to the lowest. Walk down the list, flagging firms one at a time.
- A flagged failing firm is a **catch**: step **up**. A flagged healthy firm is a **false alarm**: step **right**.
- Steps are shares: up by 1 / (number of failing firms), right by 1 / (number of healthy firms). The walk ends at the top right corner, with every firm flagged.
- A good ranking climbs first and moves right later, so its curve hugs the top left corner.
- Firms with the same score are flagged together, so their steps merge into one straight line.

In [ ]:
def roc_by_hand(failed, risk_scores):
    """Walk down the firms from the highest risk score to the lowest: step up for a failing
    firm, right for a healthy one. Return the corners of the walk: the share of healthy firms
    flagged (across) and the share of failing firms caught (up)."""
    order = np.argsort(-risk_scores, kind="stable")         # riskiest first
    failed_sorted = np.asarray(failed)[order]
    scores_sorted = np.asarray(risk_scores)[order]
    step_up = 1 / np.sum(failed_sorted == 1)       # one failing firm: its share of all failing
    step_right = 1 / np.sum(failed_sorted == 0)    # one healthy firm: its share of all healthy
    false_alarms, catches = 0.0, 0.0
    corners_across, corners_up = [0.0], [0.0]
    for position in range(len(scores_sorted)):
        if failed_sorted[position] == 1:
            catches = catches + step_up              # a catch: step up
        else:
            false_alarms = false_alarms + step_right # a false alarm: step right
        # Mark a corner once every firm with this score is flagged (equal scores go together).
        last_firm = position == len(scores_sorted) - 1
        if last_firm or scores_sorted[position + 1] != scores_sorted[position]:
            corners_across.append(false_alarms)
            corners_up.append(catches)
    return corners_across, corners_up


def area_under(corners_across, corners_up):
    """The area under the walk: add up the strips between neighboring corners,
    each one its width times its average height."""
    area = 0.0
    for k in range(1, len(corners_across)):
        width = corners_across[k] - corners_across[k - 1]
        height = (corners_up[k] + corners_up[k - 1]) / 2
        area = area + width * height
    return area

**A small example first.**

- Eight made-up firms, sorted from the highest risk score to the lowest.
- Three of them failed (`failed` = 1), five were healthy.

In [ ]:
toy = pd.DataFrame({"risk score": [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2],
                    "failed": [1, 0, 1, 0, 0, 1, 0, 0]},
                   index=["A", "B", "C", "D", "E", "F", "G", "H"])
toy

**The figure.** The walk for the eight firms. Each letter marks where the walk is after that firm.

- A failed: up a third. B was healthy: right a fifth. C failed: up. And so on to H, at the top right corner.
- The shaded part is the area under the curve. The dotted diagonal is a coin flip: a ranking that knows nothing.

In [ ]:
toy_across, toy_up = roc_by_hand(toy["failed"].to_numpy(), toy["risk score"].to_numpy())

plt.figure(figsize=(6.5, 6.2))
plt.fill_between(toy_across, toy_up, color=BLUE, alpha=0.15, label="the area under the curve")
plt.plot(toy_across, toy_up, color=BLUE, marker="o", linewidth=3, label="the walk")
for letter, across, up in zip(toy.index, toy_across[1:], toy_up[1:]):
    plt.text(across + 0.02, up - 0.06, letter, fontsize=16, color=BLUE)
plt.plot([0, 1], [0, 1], color=GRAY, linestyle=":", linewidth=2, label="a coin flip")
plt.xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
plt.yticks([0, 1 / 3, 2 / 3, 1], ["0", "1/3", "2/3", "1"])
plt.xlabel("share of healthy firms flagged")
plt.ylabel("share of failing firms caught")
plt.legend(loc="lower right", frameon=False, fontsize=14)
plt.gca().set_aspect("equal")
plt.show()

**The AUC two ways.** AUC is the **area under the ROC curve**. It is also a chance:

- Pick one failing firm and one healthy firm at random. The AUC is the chance that the failing firm has the higher risk score (a tie counts as half).
- The eight firms make 3 × 5 = 15 failing–healthy pairs. Count the pairs the ranking gets right, and compare with the area.

In [ ]:
def share_of_pairs_ranked_right(failed, risk_scores):
    """Pair every failing firm with every healthy firm. A pair is a win when the failing firm
    has the higher risk score, half a win on a tie. Return wins / pairs."""
    failing_scores = risk_scores[failed == 1].tolist()
    healthy_scores = risk_scores[failed == 0].tolist()
    wins = 0.0
    for failing_score in failing_scores:
        for healthy_score in healthy_scores:
            if failing_score > healthy_score:
                wins = wins + 1
            elif failing_score == healthy_score:
                wins = wins + 0.5
    return wins / (len(failing_scores) * len(healthy_scores))


toy_failed, toy_scores = toy["failed"].to_numpy(), toy["risk score"].to_numpy()
print(f"area under the walk:          {area_under(toy_across, toy_up):.4f}")
print(f"share of pairs ranked right:  {share_of_pairs_ranked_right(toy_failed, toy_scores):.4f}")
print(f"11 of 15 pairs, by hand:      {11 / 15:.4f}")

**Now the bankruptcy tree, on the held-out firms.**

- The same loop draws the tree's ROC curve: blue, with a dot at each corner. A tree has one corner per leaf.
- scikit-learn draws it in one line with `roc_curve`: orange dashed, on top. The two should lie on top of each other.
- The dotted diagonal is a coin flip (AUC 0.5). The closer the curve hugs the top left corner, the better the ranking.

In [ ]:
corners_across, corners_up = roc_by_hand(failed_test, risk_scores)
# scikit-learn in one line: shares of healthy firms flagged, shares of failing firms caught.
sklearn_across, sklearn_up, cutoffs = roc_curve(failed_test, risk_scores)

plt.figure(figsize=(7.5, 7))
plt.fill_between(corners_across, corners_up, color=BLUE, alpha=0.12)
plt.plot(corners_across, corners_up, color=BLUE, marker="o", linewidth=5, label="our loop")
plt.plot(sklearn_across, sklearn_up, color=VERMILLION, linestyle="--", linewidth=2.5,
         label="scikit-learn's roc_curve")
plt.plot([0, 1], [0, 1], color=GRAY, linestyle=":", linewidth=2)
plt.text(0.62, 0.55, "a coin flip (AUC 0.5)", color=GRAY, rotation=45, ha="center", va="center")
plt.xlabel("share of healthy firms flagged\n(false alarms)")
plt.ylabel("share of failing firms caught")
plt.title("one tree, firms held out, three years ahead")
plt.legend(loc="lower right", frameon=False)
plt.gca().set_aspect("equal")
plt.show()

- The textbook names of the axes: the **false positive rate** (across) and the **true positive rate** (up).
- Now the AUC three ways: the area under our loop's curve, the share of failing–healthy pairs ranked right, and scikit-learn's `roc_auc_score`.

In [ ]:
print(f"area under our loop's curve:  {area_under(corners_across, corners_up):.4f}")
print(f"share of pairs ranked right:  {share_of_pairs_ranked_right(failed_test, risk_scores):.4f}")
print(f"scikit-learn's roc_auc_score: {roc_auc_score(failed_test, risk_scores):.4f}")

# Why held-out firms: on the firms it learned from, the same tree looks better than it is.
training_scores = risk_tree.predict_proba(ratios_train)[:, 1]
print(f"AUC on its own training firms: {roc_auc_score(failed_train, training_scores):.4f}")

- All three agree: the AUC is the area under the curve, and the chance that a failing firm outranks a healthy one.
- On its own training firms the tree scores higher. That number flatters it; the held-out firms give the honest grade.

## What to Take Away

- A tree asks yes/no questions. Each question is one cut, and each leaf forecasts the average of its training rows (for a yes/no outcome: the share that failed).
- Depth and the number of boosted trees are dials. **Train fits, validation picks, test grades once.**
- Boosting adds many small trees, each fixing part of what the trees before it still get wrong.
- When few firms fail, accuracy misleads. Rank by risk instead: the ROC curve walks down the ranking, and its area, the AUC, is the chance that a failing firm outranks a healthy one. 0.5 is a coin flip.